In [9]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

In [10]:
seed = 42

root_path = "/home/stefan/ioai-prep/kits/contest/rabbit"

# Data

In [11]:
df = pd.read_csv(f"{root_path}/train_data.csv")
df_test = pd.read_csv(f"{root_path}/test_data.csv")

df.head()

,ID,Sex,Greutate,Lungime_urechi,Ureche_lăsată,Culoare,Vârstă,Tip_blană,Calitate_blană,Formă_corp,Apare_gușa,Sănătate,Scor_jurizare
0,542,Femelă,1.6,19.1,False,Havana,9,Scurtă,Excelentă,Incorectă,False,Probleme_grave_de_sănătate,0
1,441,Mascul,2.7,22.1,False,Agouti,18,Lungă,Excelentă,Aproape_corectă,False,Probleme_grave_de_sănătate,0
2,483,Femelă,7.7,42.3,True,Alb,10,Lungă,Foarte_bună,Corectă,False,Ușoare_probleme_de_sănătate,93
3,423,Femelă,1.8,19.8,False,Havana,10,Scurtă,Foarte_bună,Incorectă,False,Probleme_grave_de_sănătate,0
4,779,Femelă,7.7,42.4,True,Havana,11,Lungă,Foarte_bună,Incorectă,True,Aspect_neîngrijit,82


# Subtask 1

In [39]:
mask = (
    (df_test["Sex"] == "Femelă")
    & (df_test["Ureche_lăsată"] == 1)
    & (df_test["Culoare"] == "Havana")
)
subtask1_ans = mask.sum()
print("Subtask 1:", subtask1_ans)

Subtask 1: 11


# Subtask 2

In [ ]:
features = [
    "Sex",
    "Greutate",
    "Lungime_urechi",
    "Ureche_lăsată",
    "Culoare",
    "Vârstă",
    "Tip_blană",
    "Calitate_blană",
    "Formă_corp",
    "Apare_gușa",
    "Sănătate",
]
X_clust = df[features].copy()
X_clust = pd.get_dummies(
    X_clust,
    columns=["Sex", "Culoare", "Tip_blană", "Calitate_blană", "Formă_corp", "Sănătate"],
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)
kmeans = KMeans(n_clusters=3, random_state=seed)
clusters = kmeans.fit_predict(X_scaled)

X_test_clust = df_test[features].copy()
X_test_clust = pd.get_dummies(
    X_test_clust,
    columns=["Sex", "Culoare", "Tip_blană", "Calitate_blană", "Formă_corp", "Sănătate"],
)
X_test_clust = X_test_clust.reindex(columns=X_clust.columns, fill_value=0)
X_test_scaled = scaler.transform(X_test_clust)
subtask2 = kmeans.predict(X_test_scaled)

# Subtask 3

In [16]:
train_mask = df["Scor_jurizare"].notna()
y = df.loc[train_mask, "Scor_jurizare"]

numeric = ["Greutate", "Lungime_urechi", "Vârstă"]
categorical = [
    "Sex",
    "Ureche_lăsată",
    "Culoare",
    "Tip_blană",
    "Calitate_blană",
    "Formă_corp",
    "Apare_gușa",
    "Sănătate",
]

pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ]
)


def evaluate(model):
    pipe = Pipeline(steps=[("prep", pre), ("model", model)])
    cv = cross_val_score(
        pipe,
        df[train_mask][numeric + categorical],
        y,
        scoring="neg_root_mean_squared_error",
        cv=3,
        n_jobs=-1,
    )
    cv_score = -(cv.mean() - cv.std()).item()
    pipe.fit(df[train_mask][numeric + categorical], y)
    preds = pipe.predict(df_test[numeric + categorical])
    return cv_score, preds

In [17]:
rf = RandomForestRegressor(random_state=seed)
cv_rf, preds_rf = evaluate(rf)
print("RF cv:", cv_rf)

RF cv: 1.159769559147384


In [18]:
gb = GradientBoostingRegressor(random_state=seed)
cv_gb, preds_gb = evaluate(gb)
print("GB cv:", cv_gb)

GB cv: 2.434725048541164


In [32]:
best_preds = preds_rf
subtask3 = best_preds

# Submission

In [40]:
def build_subtask(sid, answer):
    if sid == 1:
        return pd.DataFrame({
            "subtaskID": sid,
            "datapointID": [1],
            "answer": answer
        })
    else:
        return pd.DataFrame({
            "subtaskID": sid,
            "datapointID": df_test["ID"],
            "answer": answer
        })

subtasks = [(1, subtask1_ans), (2, subtask2), (3, subtask3)]
submission = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])
submission.to_csv(f"{root_path}/submission.csv", index=False)

In [41]:
submission.head()

,subtaskID,datapointID,answer
0,1,1,11.0
0,2,522,2.0
1,2,738,2.0
2,2,741,1.0
3,2,661,0.0
